In [12]:
from nero_interface_server import NeroDualArmServer

In [13]:
import logging
log = logging.getLogger(__name__)

In [14]:
import numpy as np
import time

In [15]:
server = NeroDualArmServer(gripper_enabled=True)

[SERVER] Failed to connect to right arm: CAN socket can_right does not exist.


[Left Arm] 正在获取当前关节角作为 IK 初始基准...
[Left Arm] IK Solver 初始化完成！初始关节角: [ 1.037  0.011 -1.093  2.064  0.172  0.343 -0.049]


In [16]:
fp = server.left_robot.get_flange_pose()
robot_pose = np.array(fp.msg)

print("robot z:", robot_pose[2])

robot z: 0.23954699999999998


In [17]:
assert server.left_robot is not None, "Left arm failed"
assert server.right_robot is not None, "Right arm failed"
print("Left robot:", server.left_robot)
print("Right robot:", server.right_robot)

Left robot: <pyAgxArm.protocols.can_protocol.drivers.nero.default.driver.Driver object at 0x7f2938e6f9d0>
Right robot: <pyAgxArm.protocols.can_protocol.drivers.nero.default.driver.Driver object at 0x7f2937f2f3d0>


In [18]:
left_joints = server.left_robot_get_joint_positions()
right_joints = server.right_robot_get_joint_positions()

print("Left joints:", left_joints)
print("Right joints:", right_joints)

Left joints: [1.0368302954397515, 0.010663961729685353, -1.0933615099118479, 2.063816933898255, 0.17158331876356253, 0.34269539862908666, -0.04876449930072157]
Right joints: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


In [19]:
left_pose = server.left_robot_get_ee_pose()
right_pose = server.right_robot_get_ee_pose()
print("Left pose:", left_pose)
print("Right pose:", right_pose)

Left pose: [-0.378691, 0.079886, 0.23954699999999998, 1.6022297066233144, -0.47148324413374815, -0.452144996021651]
Right pose: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


In [20]:
left_arm_status = server.left_robot_get_arm_status()
right_arm_status = server.right_robot_get_arm_status()

print("Left arm status:", left_arm_status)
print("Right arm status:", right_arm_status)

Left arm status: {'ctrl_mode': CAN_CTRL(0x1), 'arm_status': NORMAL(0x0), 'motion_status': REACH_TARGET_POS_FAILED(0x1), 'trajectory_num': 0}
Right arm status: {'ctrl_mode': 0, 'arm_status': 0, 'motion_status': 0}


In [38]:
server.left_robot_go_home()

left_joints = server.left_robot_get_joint_positions()
print("Left joints:", left_joints)

Left joints: [8.726646259971648e-05, -0.12997466939601773, -1.7453292519943296e-05, 1.8699632138792448, 0.0, -3.490658503988659e-05, -0.16994270926668786]


In [32]:
tcp = server.left_robot.get_tcp_pose()
print(tcp.msg)

[-0.398184, 0.03558, 0.24267899999999998, 1.5668693359779096, -0.39128536500460875, -0.0859400123682008]


In [11]:
# left_joints = server.left_robot_get_joint_positions()
# print("Left joints:", left_joints)

# left_joints_target = np.array(left_joints).copy()
# left_joints_target[0] -= np.deg2rad(0)
# left_joints_target[1] += np.deg2rad(0)
# left_joints_target[2] += np.deg2rad(0)
# left_joints_target[3] += np.deg2rad(0)
# left_joints_target[4] += np.deg2rad(0)
# left_joints_target[5] += np.deg2rad(0)
# left_joints_target[6] += np.deg2rad(0)

# print("Left joints target:", left_joints_target)

# server.left_robot_move_to_joint_positions(left_joints_target, delta=False)

# left_joints_real = server.left_robot_get_joint_positions()
# print("Left joints:", left_joints_real)

In [12]:
# left_pose = server.left_robot_get_ee_pose()
# print("Left pose:", left_pose)

# pose_target = np.array(left_pose).copy()
# # end-effector pose [x, y, z, roll, pitch, yaw] (m, rad)
# # pose_target[0] += 0.01   # x方向移动1cm
# # pose_target[1] += 0.01
# # pose_target[2] += 0.01
# pose_target = [-0.4, 0.0, 0.4, 1.5708, 0.0, 0.0]

# server.left_robot_move_to_ee_pose(pose_target)
# time.sleep(3.0)
# left_pose = server.left_robot_get_ee_pose()
# print("Left pose:", left_pose)

In [13]:
# 绝对控制
# target = [0, -70, 0, 150, 0, 0, 30]  # 7维绝对关节角度（度）

# ok = server.servo_j("left_robot", target, delta=False)
# print("servo_j ok:", ok)

In [14]:
# 增量控制
# target_delta = np.zeros(7)
# target_delta[0] = 10

# ok = server.servo_j("left_robot", target_delta.tolist(), delta=True)
# print("servo_j_delta ok:", ok)

In [19]:
# 当前位姿
fp = server.left_robot.get_flange_pose()
current_pose = np.array(fp.msg, dtype=float)
print("ipynb当前位姿:", current_pose)

# 构造一个小的 delta
delta_pose = np.array([0.05, 0.0, 0.0, 0.0, 0.0, 0.0])
# target_pose = np.array([-0.5, 0.05, 0.2, 1.46, -0.7, -0.04])

# 调用 servo_p（delta 模式）
ret = server.servo_p(
    robot_arm="left_robot",
    pose=delta_pose.tolist(),
    delta=True
    # pose=target_pose.tolist(),
    # delta=False
)

time.sleep(1.5)

fp = server.left_robot.get_flange_pose()
new_pose = np.array(fp.msg, dtype=float)
print("新位姿:", new_pose)

ipynb当前位姿: [-4.00956000e-01  1.60000000e-05  4.00029000e-01  1.57079633e+00
  7.50491578e-04 -8.72664626e-05]
当前位姿: [-4.00958000e-01 -1.20000000e-05  4.00620000e-01  1.57077887e+00
  4.01425728e-03  5.23598776e-05]
目标姿态 XYZ: [-3.50958e-01 -1.20000e-05  4.00620e-01]
目标姿态 RPY: (1.57077887, 0.00401425728, 5.235987759999991e-05)
⚠️ IK 求解失败: continuous_global_fallback
   目标位姿: x=-0.351, y=-0.000, z=0.401
   候选解数量: 0
计算出的关节角度: [ 0.         -0.12999212  0.          1.86999812  0.          0.
 -0.16999507]
新位姿: [-4.01971000e-01 -1.67000000e-04  3.84994000e-01  1.57076142e+00
 -3.87288561e-02  4.53785606e-04]


In [18]:
# 当前位姿
fp = server.left_robot.get_flange_pose()
current_pose = np.array(fp.msg, dtype=float)
# print("当前位姿:", current_pose)

target_pose = current_pose.copy()
# target_pose[0] += 0.01
# print("目标位姿:", target_pose)

steps = 10
dt = 0.02  # 20Hz

# for i in range(steps):
#     alpha = (i + 1) / steps
#     pose_i = current_pose * (1 - alpha) + target_pose * alpha

#     server.servo_p("left_robot", pose_i.tolist(), delta=False)
#     time.sleep(dt)

for i in range(steps):
    # target_pose[2] += 0.02 / steps   # 每次走一点点
    delta_pose = np.array([-0.01, 0.0, 0.0, 0.0, 0.0, 0.0])
    time1 = time.time()
    server.servo_p("left_robot", delta_pose.tolist(), delta=True)
    time2 = time.time()
    print(f"Step {i+1}/{steps}, servo_p time: {(time2 - time1) * 1000:.2f} ms")
    time.sleep(0.005)

当前位姿: [-2.25957361e-01  1.91592450e-05  3.99901661e-01 -1.57076756e+00
 -1.15366214e-02  3.14137566e+00]
目标姿态 XYZ: [-2.35957361e-01  1.91592450e-05  3.99901661e-01]
目标姿态 RPY: (-1.5707675563233976, -0.011536621419923114, 3.1413756605684937)
计算出的关节角度: [-1.83098907e-02 -9.68851866e-02  1.65313177e-02  1.84129719e+00
  1.32013291e-03 -1.89599533e-03 -1.85165487e-01]
Step 1/10, servo_p time: 564.58 ms
当前位姿: [-2.25957362e-01 -5.59254633e-07  3.99901661e-01 -1.57073521e+00
 -1.15366299e-02 -3.14153351e+00]
目标姿态 XYZ: [-2.35957362e-01 -5.59254633e-07  3.99901661e-01]
目标姿态 RPY: (-1.57073521313794, -0.011536629907542882, -3.1415335106618074)
计算出的关节角度: [-1.82263226e-02 -9.68851864e-02  1.65313178e-02  1.84129719e+00
  1.31865561e-03 -2.09162251e-03 -1.85165490e-01]
Step 2/10, servo_p time: 97.67 ms
当前位姿: [-2.25957360e-01 -2.81651541e-05  3.99901661e-01 -1.57083300e+00
 -1.14144532e-02 -3.14146433e+00]
目标姿态 XYZ: [-2.35957360e-01 -2.81651541e-05  3.99901661e-01]
目标姿态 RPY: (-1.5708329993374313, -0.01

In [ ]:
# server.left_gripper_goto(0.05)
# # server.left_gripper_grasp()
# print(server.left_gripper_get_state())

In [ ]:
# server.stop("left_robot")